# 10 - Modeling v3: Uji CatBoost dengan Fitur Rasio + Recent Window

Menguji apakah 165 fitur baru (rasio custom + recent-window 6 bulan) dari
Feature Engineering v3 meningkatkan performa CatBoost dibanding v2 (CV
ROC-AUC 0.7795), setelah ensemble dan hyperparameter tuning terbukti mentok.

# Import

In [1]:
import sys
import json
import numpy as np
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import config
from modeling.train import (
    load_featured_dataset, create_or_load_holdout_split, create_or_load_cv_folds,
    train_catboost,train_xgboost,train_lightgbm,evaluate_predictions,train_stacking_meta_model,
    save_stacking_meta_model,save_oof_predictions,
)

## Load Dataset v3, Metadata v3, dan Split/Folds yang Sudah Ada

`holdout_split.json`/`cv_folds.json` tetap dipakai ulang -- SK_ID_CURR dan
jumlah baris v3 identik dengan v1/v2, jadi split tetap adil untuk perbandingan.

In [2]:
df_v3 = load_featured_dataset(path=config.PROCESSED_DATA_DIR / "application_train_featured_v3.csv")

with open(config.PROCESSED_DATA_DIR / "feature_engineering_metadata_v3.json") as f:
    metadata_v3 = json.load(f)

split = create_or_load_holdout_split(df_v3)
folds = create_or_load_cv_folds(df_v3, split)

print("Shape dataset v3:", df_v3.shape)
print("Jumlah tree_features_v3:", len(metadata_v3["tree_features_v3"]))

Shape dataset v3: (307511, 746)
Jumlah tree_features_v3: 739


## Retrain CatBoost dengan Fitur v3

`model_name="catboost_v3"` supaya hasil tersimpan terpisah dari CatBoost v2
(default maupun tuned).

In [3]:
metadata_for_catboost_v3 = {"tree_features": metadata_v3["tree_features_v3"]}

result_cb_v3 = train_catboost(df_v3, metadata_for_catboost_v3, split, folds, model_name="catboost_v3")

print("=== CatBoost v3 ===")
print("CV ROC-AUC: {:.4f} ± {:.4f}".format(
    result_cb_v3["cv_summary"]["roc_auc_mean"], result_cb_v3["cv_summary"]["roc_auc_std"]
))
print("CV PR-AUC : {:.4f} ± {:.4f}".format(
    result_cb_v3["cv_summary"]["pr_auc_mean"], result_cb_v3["cv_summary"]["pr_auc_std"]
))

print("\n--- Perbandingan ---")
print("CatBoost v2 (default): ROC-AUC 0.7795 | PR-AUC 0.2690")
print("CatBoost v2 (tuned)  : ROC-AUC 0.7795 | PR-AUC 0.2684")
print("CatBoost v1          : ROC-AUC 0.7594 | PR-AUC 0.2436")

d:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\src\modeling\train.py:674: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dev_df = df.set_index(id_column).loc[dev_ids].reset_index()


=== CatBoost v3 ===
CV ROC-AUC: 0.7829 ± 0.0020
CV PR-AUC : 0.2720 ± 0.0056

--- Perbandingan ---
CatBoost v2 (default): ROC-AUC 0.7795 | PR-AUC 0.2690
CatBoost v2 (tuned)  : ROC-AUC 0.7795 | PR-AUC 0.2684
CatBoost v1          : ROC-AUC 0.7594 | PR-AUC 0.2436


## Retrain LightGBM & XGBoost dengan Fitur v3

Menguji apakah kenaikan skor CatBoost v3 (0.7829) juga berlaku untuk model
boosting lain, sekaligus menyiapkan bahan untuk ensemble ulang.

In [4]:
metadata_for_boosting_v3 = {"tree_features": metadata_v3["tree_features_v3"]}

result_lgb_v3 = train_lightgbm(df_v3, metadata_for_boosting_v3, split, folds, model_name="lightgbm_v3")
print("=== LightGBM v3 ===")
print("CV ROC-AUC: {:.4f} ± {:.4f}".format(
    result_lgb_v3["cv_summary"]["roc_auc_mean"], result_lgb_v3["cv_summary"]["roc_auc_std"]
))
print("CV PR-AUC : {:.4f} ± {:.4f}".format(
    result_lgb_v3["cv_summary"]["pr_auc_mean"], result_lgb_v3["cv_summary"]["pr_auc_std"]
))
print("(LightGBM v2: ROC-AUC 0.7775 | PR-AUC 0.2664)\n")

result_xgb_v3 = train_xgboost(df_v3, metadata_for_boosting_v3, split, folds, model_name="xgboost_v3")
print("=== XGBoost v3 ===")
print("CV ROC-AUC: {:.4f} ± {:.4f}".format(
    result_xgb_v3["cv_summary"]["roc_auc_mean"], result_xgb_v3["cv_summary"]["roc_auc_std"]
))
print("CV PR-AUC : {:.4f} ± {:.4f}".format(
    result_xgb_v3["cv_summary"]["pr_auc_mean"], result_xgb_v3["cv_summary"]["pr_auc_std"]
))
print("(XGBoost v2: ROC-AUC 0.7733 | PR-AUC 0.2606)")

d:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\src\modeling\train.py:674: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dev_df = df.set_index(id_column).loc[dev_ids].reset_index()
c:\Users\USER\miniconda3\envs\Homecreadit\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\USER\miniconda3\envs\Homecreadit\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\USER\miniconda3\envs\Home

=== LightGBM v3 ===
CV ROC-AUC: 0.7810 ± 0.0020
CV PR-AUC : 0.2702 ± 0.0053
(LightGBM v2: ROC-AUC 0.7775 | PR-AUC 0.2664)



d:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\src\modeling\train.py:674: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dev_df = df.set_index(id_column).loc[dev_ids].reset_index()


=== XGBoost v3 ===
CV ROC-AUC: 0.7777 ± 0.0023
CV PR-AUC : 0.2646 ± 0.0051
(XGBoost v2: ROC-AUC 0.7733 | PR-AUC 0.2606)


## Ensemble v3 (Rata-rata OOF, Tanpa Retrain Tambahan)

Menghitung ROC-AUC/PR-AUC dari rata-rata prediksi out-of-fold CatBoost v3,
LightGBM v3, dan XGBoost v3 -- gratis secara komputasi karena memakai OOF
yang sudah tersimpan dari hasil CV.

In [5]:


oof_true_v3 = result_cb_v3["oof_true"]
assert np.allclose(oof_true_v3, result_lgb_v3["oof_true"], equal_nan=True)
assert np.allclose(oof_true_v3, result_xgb_v3["oof_true"], equal_nan=True)

oof_cb_v3 = result_cb_v3["oof_proba"]
oof_lgb_v3 = result_lgb_v3["oof_proba"]
oof_xgb_v3 = result_xgb_v3["oof_proba"]

oof_ensemble_equal_v3 = (oof_cb_v3 + oof_lgb_v3 + oof_xgb_v3) / 3
metrics_equal_v3 = evaluate_predictions(oof_true_v3, oof_ensemble_equal_v3, threshold=0.5)

print("=== Ensemble v3 Rata-rata Sederhana ===")
print("ROC-AUC:", round(metrics_equal_v3["roc_auc"], 4))
print("PR-AUC :", round(metrics_equal_v3["pr_auc"], 4))

weights_v3 = {"cb": 0.5, "lgb": 0.3, "xgb": 0.2}
oof_ensemble_weighted_v3 = (
    weights_v3["cb"] * oof_cb_v3 + weights_v3["lgb"] * oof_lgb_v3 + weights_v3["xgb"] * oof_xgb_v3
)
metrics_weighted_v3 = evaluate_predictions(oof_true_v3, oof_ensemble_weighted_v3, threshold=0.5)

print("\n=== Ensemble v3 Berbobot (CatBoost 50% / LightGBM 30% / XGBoost 20%) ===")
print("ROC-AUC:", round(metrics_weighted_v3["roc_auc"], 4))
print("PR-AUC :", round(metrics_weighted_v3["pr_auc"], 4))

print("\n--- Perbandingan Lengkap ---")
print("CatBoost v3 (sendiri) : ROC-AUC 0.7829")
print("Ensemble v2 (referensi): ROC-AUC 0.7804 (rata-rata) / berbobot ROC-AUC 0.7804")

=== Ensemble v3 Rata-rata Sederhana ===
ROC-AUC: 0.7834
PR-AUC : 0.2725

=== Ensemble v3 Berbobot (CatBoost 50% / LightGBM 30% / XGBoost 20%) ===
ROC-AUC: 0.7839
PR-AUC : 0.2735

--- Perbandingan Lengkap ---
CatBoost v3 (sendiri) : ROC-AUC 0.7829
Ensemble v2 (referensi): ROC-AUC 0.7804 (rata-rata) / berbobot ROC-AUC 0.7804


## Stacking Meta-Model (Logistic Regression di Atas OOF)

Melatih Logistic Regression yang belajar sendiri bobot optimal dari 3 prediksi
OOF (CatBoost v3, LightGBM v3, XGBoost v3), dievaluasi dengan fold yang sama
persis -- dibandingkan dengan ensemble berbobot manual (0.7839).

In [6]:
oof_true_v3 = result_cb_v3["oof_true"]
oof_proba_dict_v3 = {
    "catboost": result_cb_v3["oof_proba"],
    "lightgbm": result_lgb_v3["oof_proba"],
    "xgboost": result_xgb_v3["oof_proba"],
}

stacking_result_v3 = train_stacking_meta_model(oof_true_v3, oof_proba_dict_v3, folds)

print("=== Stacking Meta-Model v3 (Logistic Regression) ===")
print("CV ROC-AUC: {:.4f} ± {:.4f}".format(
    stacking_result_v3["cv_summary"]["roc_auc_mean"], stacking_result_v3["cv_summary"]["roc_auc_std"]
))
print("CV PR-AUC : {:.4f} ± {:.4f}".format(
    stacking_result_v3["cv_summary"]["pr_auc_mean"], stacking_result_v3["cv_summary"]["pr_auc_std"]
))

import pandas as pd
coef_df = pd.DataFrame(stacking_result_v3["fold_coefficients"])
print("\nBobot yang dipelajari (rata-rata tiap fold):")
print(coef_df.mean())

print("\n--- Perbandingan ---")
print("Ensemble v3 berbobot manual (50/30/20): ROC-AUC 0.7839")
print("CatBoost v3 (sendiri)                 : ROC-AUC 0.7829")

=== Stacking Meta-Model v3 (Logistic Regression) ===
CV ROC-AUC: 0.7841 ± 0.0020
CV PR-AUC : 0.2739 ± 0.0056

Bobot yang dipelajari (rata-rata tiap fold):
catboost    3.075642
lightgbm    1.590132
xgboost     0.348914
dtype: float64

--- Perbandingan ---
Ensemble v3 berbobot manual (50/30/20): ROC-AUC 0.7839
CatBoost v3 (sendiri)                 : ROC-AUC 0.7829


# Simpan Meta-Model

In [7]:
save_paths = save_stacking_meta_model(stacking_result_v3, model_name="stacking_meta_v3")
print("Meta-model tersimpan di:")
for key, path in save_paths.items():
    print(f"  {key}: {path}")

Meta-model tersimpan di:
  model_path: D:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\models\stacking_meta_v3\stacking_meta_v3_model.joblib
  metrics_path: D:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\models\stacking_meta_v3\stacking_meta_v3_metrics.json
  config_path: D:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\models\stacking_meta_v3\stacking_meta_v3_config.json


In [9]:
oof_path = save_oof_predictions(
    result_cb_v3["oof_true"], result_cb_v3["oof_proba"], model_name="catboost_v3",
)
print(f"OOF CatBoost v3 tersimpan di: {oof_path}")

OOF CatBoost v3 tersimpan di: D:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\models\catboost_v3\catboost_v3_oof.csv
